In [2]:
import cellxgene_census
import pandas as pd
import scanpy as sc
import numpy as np
import json
import anndata as ad


In [2]:
cell_gene_dir = "../data/censusxgene"
cell_types = [
    "malignant cell",
    "luminal epithelial cell of mammary gland",
    "basal-myoepithelial cell of mammary gland",
    "fibroblast of mammary gland",
    "macrophage",
    "T cell",
    "B cell"
]


In [3]:
with cellxgene_census.open_soma(census_version="2025-11-08") as census:
    obs_df = cellxgene_census.get_obs(
        census,
        "homo_sapiens",
        value_filter=f"tissue == 'breast' and suspension_type == 'cell' and cell_type in {cell_types}"
    )

In [4]:
obs_df

,soma_joinid,dataset_id,assay,assay_ontology_term_id,cell_type,cell_type_ontology_term_id,development_stage,development_stage_ontology_term_id,disease,disease_ontology_term_id,...,tissue,tissue_ontology_term_id,tissue_type,tissue_general,tissue_general_ontology_term_id,raw_sum,nnz,raw_mean_nnz,raw_variance_nnz,n_measured_vars
0,107899,44941fdb-8a6f-42d5-9b34-da48c2a3f774,10x 3' v2,EFO:0009899,malignant cell,CL:0001064,52-year-old stage,HsapDv:0000146,breast cancer,MONDO:0007254,...,breast,UBERON:0000310,tissue,breast,UBERON:0000310,12532.0,3097,4.046497,867.827295,16727
1,107900,44941fdb-8a6f-42d5-9b34-da48c2a3f774,10x 3' v2,EFO:0009899,malignant cell,CL:0001064,52-year-old stage,HsapDv:0000146,breast cancer,MONDO:0007254,...,breast,UBERON:0000310,tissue,breast,UBERON:0000310,19983.0,5026,3.975925,473.673649,16727
2,107901,44941fdb-8a6f-42d5-9b34-da48c2a3f774,10x 3' v2,EFO:0009899,malignant cell,CL:0001064,52-year-old stage,HsapDv:0000146,breast cancer,MONDO:0007254,...,breast,UBERON:0000310,tissue,breast,UBERON:0000310,19872.0,4868,4.082169,545.674365,16727
3,107902,44941fdb-8a6f-42d5-9b34-da48c2a3f774,10x 3' v2,EFO:0009899,malignant cell,CL:0001064,52-year-old stage,HsapDv:0000146,breast cancer,MONDO:0007254,...,breast,UBERON:0000310,tissue,breast,UBERON:0000310,19774.0,4681,4.224311,487.866768,16727
4,107903,44941fdb-8a6f-42d5-9b34-da48c2a3f774,10x 3' v2,EFO:0009899,malignant cell,CL:0001064,52-year-old stage,HsapDv:0000146,breast cancer,MONDO:0007254,...,breast,UBERON:0000310,tissue,breast,UBERON:0000310,19691.0,3784,5.203753,640.641265,16727
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2217683,134319592,55003f67-c494-46f1-83fb-902745646379,10x 3' v3,EFO:0009922,macrophage,CL:0000235,30-year-old stage,HsapDv:0000124,normal,PATO:0000461,...,breast,UBERON:0000310,tissue,breast,UBERON:0000310,4619.0,1220,3.786066,854.361906,15153
2217684,134319603,55003f67-c494-46f1-83fb-902745646379,10x 3' v3,EFO:0009922,fibroblast of mammary gland,CL:0002555,33-year-old stage,HsapDv:0000127,normal,PATO:0000461,...,breast,UBERON:0000310,tissue,breast,UBERON:0000310,2308.0,1203,1.918537,15.871894,15153
2217685,134319605,55003f67-c494-46f1-83fb-902745646379,10x 3' v3,EFO:0009922,macrophage,CL:0000235,46-year-old stage,HsapDv:0000140,normal,PATO:0000461,...,breast,UBERON:0000310,tissue,breast,UBERON:0000310,5864.0,1644,3.566910,160.392964,15153
2217686,134319606,55003f67-c494-46f1-83fb-902745646379,10x 3' v3,EFO:0009922,fibroblast of mammary gland,CL:0002555,48-year-old stage,HsapDv:0000142,normal,PATO:0000461,...,breast,UBERON:0000310,tissue,breast,UBERON:0000310,11659.0,2524,4.619255,390.625091,15153


In [18]:
cells_per_type = 4000
selected_ids = []

for ct in cell_types:
    subset = obs_df[obs_df["cell_type"] == ct]
    n_available = len(subset)
    
    n_sample = min(cells_per_type, n_available)
    
    ids = np.random.choice(subset["soma_joinid"].values, n_sample, replace=False)
    selected_ids.extend(ids)

    print(ct, len(subset), n_sample)

malignant cell 556411 4000
luminal epithelial cell of mammary gland 242832 4000
basal-myoepithelial cell of mammary gland 308713 4000
fibroblast of mammary gland 833781 4000
macrophage 142945 4000
T cell 115565 4000
B cell 17441 4000


In [19]:
with cellxgene_census.open_soma(census_version="2025-11-08") as census:
    cell_adata = cellxgene_census.get_anndata(
        census,
        "homo_sapiens",
        obs_coords=selected_ids,
        value_filter=f"tissue == 'breast' and suspension_type == 'cell' and cell_type in {cell_types}",
        column_names=["assay", "cell_type", "tissue", "tissue_general", "suspension_type", "disease"]
    )

/tmp/ipykernel_1992033/2780810093.py:2: FutureWarning: The argument `column_names` is deprecated and will be removed in a future release. Please use `obs_column_names` and `var_column_names` instead.
  cell_adata = cellxgene_census.get_anndata(
/scratch/2370352/conda/envs/myenv/lib/python3.9/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
/scratch/2370352/conda/envs/myenv/lib/python3.9/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


In [24]:
cell_adata.obs['cell_type'].value_counts()[:10]

cell_type
fibroblast of mammary gland                     4000
luminal epithelial cell of mammary gland        4000
B cell                                          4000
basal-myoepithelial cell of mammary gland       4000
T cell                                          4000
macrophage                                      4000
malignant cell                                  4000
mucosa-associated lymphoid tissue macrophage       0
motor neuron                                       0
mononuclear phagocyte                              0
Name: count, dtype: int64

In [25]:
sc.pp.normalize_total(cell_adata, target_sum=1e4, inplace=False)
sc.pp.log1p(cell_adata)

In [25]:
cell_adata.write("../../data/censusxgene/breast_norm_selected.h5ad")


In [26]:
print(cell_adata)
print("obs columns:", cell_adata.obs.columns)
print("var shape:", cell_adata.var.shape)
print("X type:", type(cell_adata.X))

AnnData object with n_obs × n_vars = 28000 × 61497
    obs: 'soma_joinid', 'dataset_id', 'assay', 'assay_ontology_term_id', 'cell_type', 'cell_type_ontology_term_id', 'development_stage', 'development_stage_ontology_term_id', 'disease', 'disease_ontology_term_id', 'donor_id', 'is_primary_data', 'observation_joinid', 'self_reported_ethnicity', 'self_reported_ethnicity_ontology_term_id', 'sex', 'sex_ontology_term_id', 'suspension_type', 'tissue', 'tissue_ontology_term_id', 'tissue_type', 'tissue_general', 'tissue_general_ontology_term_id', 'raw_sum', 'nnz', 'raw_mean_nnz', 'raw_variance_nnz', 'n_measured_vars'
    var: 'soma_joinid', 'feature_id', 'feature_name', 'feature_type', 'feature_length', 'nnz', 'n_measured_obs'
    uns: 'log1p'
obs columns: Index(['soma_joinid', 'dataset_id', 'assay', 'assay_ontology_term_id',
       'cell_type', 'cell_type_ontology_term_id', 'development_stage',
       'development_stage_ontology_term_id', 'disease',
       'disease_ontology_term_id', 'donor_id

In [27]:
vocab_path = '/scratch/2370352/my-research/papers/scgpt/save/whole_human/vocab.json'

with open(vocab_path, "r") as f:
    vocab = json.load(f)

model_genes = list(vocab.keys())

print(f"Liczba genów w scGPT vocab: {len(model_genes)}")
print(model_genes[:10])  # przykładowe pierwsze 10 genów

Liczba genów w scGPT vocab: 60697
['RP5-973N23.5', 'RP11-182N22.10', 'CTB-53D8.3', 'RP11-348N17.2', 'RP11-205M20.8', 'RP11-326C3.17', 'RP11-439H13.3', 'RP11-413H22.3', 'GET1-SH3BGR', 'CH17-476P10.1']


In [28]:
# adata_genes = lista genów z adata
# model_genes = lista genów z scGPT vocab

adata_genes = cell_adata.var['feature_name'].tolist()

# zamień na sety dla szybkiego porównania
adata_set = set(adata_genes)
model_set = set(model_genes)

# wspólne geny
common_genes = adata_set & model_set

# geny w adata, których nie ma w scGPT
missing_in_model = adata_set - model_set

# geny w scGPT, których nie ma w adata
missing_in_adata = model_set - adata_set

print(f"Liczba genów w adata: {len(adata_genes)}")
print(f"Liczba genów w scGPT vocab: {len(model_genes)}")
print(f"Liczba genów wspólnych: {len(common_genes)}")
print(f"Liczba genów w adata nie w vocab: {len(missing_in_model)}")
print(f"Liczba genów w vocab nie w adata: {len(missing_in_adata)}")

# przykładowe geny
print("Przykłady genów wspólnych:", list(common_genes)[:10])
print("Przykłady genów w adata ale nie w vocab:", list(missing_in_model)[:10])
print("Przykłady genów w vocab ale nie w adata:", list(missing_in_adata)[:10])

Liczba genów w adata: 61497
Liczba genów w scGPT vocab: 60697
Liczba genów wspólnych: 38598
Liczba genów w adata nie w vocab: 21430
Liczba genów w vocab nie w adata: 22099
Przykłady genów wspólnych: ['PRB2', 'PTTG2', 'SPATA2P1', 'SLC1A2', 'BRF1', 'TTC31', 'SNX18P12', 'ANXA9', 'ETF1P2', 'MIR648']
Przykłady genów w adata ale nie w vocab: ['ENSG00000251058', 'ENSG00000260171', 'ENSG00000255210', 'ENSG00000290112', 'ENSG00000273199', 'ENSG00000260209', 'ENSG00000273760', 'ENSG00000275231', 'ENSG00000258698', 'ENSG00000266302']
Przykłady genów w vocab ale nie w adata: ['Y_RNA_ENSG00000202414', 'RP11-335J9.1', 'RP11-17A19.2', 'RP11-63N3.1', 'LINC01115_ENSG00000272342', 'RP11-15L13.5', 'CTD-2311M21.4', 'ENSG00000221498.1', 'RP5-930J4.2', 'CTD-2313J17.6']


In [29]:
gene_info = pd.read_csv("/scratch/2370352/my-research/data/gene_info_table.csv")  # lub pełna ścieżka

# Stwórz słownik: ensembl_id -> gene_name
ensg_to_symbol = dict(zip(gene_info['ensembl_id'], gene_info['gene_name']))

print(list(ensg_to_symbol.items())[:10])

[('ENSG00000000003', 'TSPAN6'), ('ENSG00000000005', 'TNMD'), ('ENSG00000000419', 'DPM1'), ('ENSG00000000457', 'SCYL3'), ('ENSG00000000460', 'C1orf112'), ('ENSG00000000938', 'FGR'), ('ENSG00000000971', 'CFH'), ('ENSG00000001036', 'FUCA2'), ('ENSG00000001084', 'GCLC'), ('ENSG00000001167', 'NFYA')]


In [30]:
# jeśli w adata masz geny zapisane jako ENSG, mapujemy je
mapped_genes = []

for g in cell_adata.var['feature_name']:
    if g in ensg_to_symbol:
        mapped_genes.append(ensg_to_symbol[g])
    else:
        mapped_genes.append(g)  # zachowaj tak jak jest, np. już symboliczny gen

# podmieniamy w adata.var
cell_adata.var['feature_name_mapped'] = mapped_genes

In [31]:
np.random.seed(42)

n = cell_adata.n_obs
indices = np.arange(n)

np.random.shuffle(indices)

train_size = int(0.8 * n)

train_idx = indices[:train_size]
test_idx  = indices[train_size:]

train_adata = cell_adata[train_idx].copy()
test_adata  = cell_adata[test_idx].copy()

In [32]:
train_adata.write("/scratch/2370352/my-research/adapter_premium/data_new/train.h5ad")
test_adata.write("/scratch/2370352/my-research/adapter_premium/data_new/test.h5ad")


In [3]:
adata = ad.read_h5ad("/scratch/2370352/my-research/adapter_premium/data_new/train.h5ad")


In [5]:
adata.obs['cell_type'].value_counts()

cell_type
T cell                                       3237
malignant cell                               3225
macrophage                                   3210
fibroblast of mammary gland                  3203
luminal epithelial cell of mammary gland     3185
B cell                                       3176
basal-myoepithelial cell of mammary gland    3164
Name: count, dtype: int64

In [6]:
adata_test = ad.read_h5ad("/scratch/2370352/my-research/adapter_premium/data_new/test.h5ad")
adata_test.obs['cell_type'].value_counts()


cell_type
basal-myoepithelial cell of mammary gland    836
B cell                                       824
luminal epithelial cell of mammary gland     815
fibroblast of mammary gland                  797
macrophage                                   790
malignant cell                               775
T cell                                       763
Name: count, dtype: int64